# NY Lottery Sales Investigation

## About the Project

### Goals

1. Find how many NY Lottery retailers are in each NY state school district.

### Libraries


- Pandas
- Geopandas
- matplotlib

### Process

1. Get school district locations for retailers
    - List of retailers: (https://data.ny.gov/Government-Finance/NYS-Lottery-Retailers/2vvn-pdyi/about_data)
    - List of school districts: (https://edit.nylottery.ny.gov/sites/default/files/2026-08/Aid%20to%20Education%20FY25-26%20Final%20%281%29.pdf)
2. Group retailers together by school district
3. Map retail locations and school districts

### Data Sources

- Lottery Sales
    - Since 2024:
        - https://data.ny.gov/Government-Finance/Lottery-Daily-Retailer-Sales-By-Game-Beginning-202/xyvi-fbb9/about_data
    - 2022 - 2023
        - https://data.ny.gov/Government-Finance/Lottery-Annual-Retailer-Sales-By-Game-2022-2023/wyec-3ji4/about_data
- NY Lottery Retailers
    - https://data.ny.gov/Government-Finance/NYS-Lottery-Retailers/2vvn-pdyi/about_data
- School Districts
    - https://data.gis.ny.gov/datasets/school-districts/explore?location=42.704583%2C-75.823549%2C6


## Part 0: Import Libraries

In [1]:
import pandas as pd
import geopandas as gpd
import geodatasets
import matplotlib.pyplot as plt

%matplotlib inline

----

## I. Import files

### 1. Import NY School Districts as "SD"

In [2]:
sd = gpd.read_file("../Data/nys_schools")
sd.head(1)

,SCHOOL_ID,SCHOOLDIST,POLYTYPE,PRIMARYPOL,USERNAME,UPDATE_,EDITED,EACODE,District2,Inst_ID,...,SEDdir_B_1,SED_CODE_1,POPULAR_NA,INSTPECD,INSTSUBYPE,INSSUBDE,MUNICODE,SDLCODE,ORPTSCOD_1,geometry
0,288,SO MOUNTAIN,DTM ONLY,Y,unknown,1900-01-01,NONE,032201,SO MOUNTAIN,800000054990,...,NaN,030201080000,SOUTH MOUNTAIN-HICKORY COMN SD AT BI,16.0,10.0,100% CONTRACT,031007300100,030201,032201,"POLYGON ((423174.308 4658625.221, 423127.151 4..."


Filter out specific columns:
- SCHOOLDIST, DISTRICT2, SCHDIST, POPULAR_NA, GEOMETRY

In [3]:
sd = sd[['SCHOOLDIST', 'geometry']]

In [4]:
sd.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 936 entries, 0 to 935
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   SCHOOLDIST  936 non-null    str     
 1   geometry    936 non-null    geometry
dtypes: geometry(1), str(1)
memory usage: 14.8 KB


### 2. Import NY retailers as "nyr"

In [5]:
retail = pd.read_csv("../Data/nys_retailers_2026.csv")
retail.head()

,Retailer,Name,Street,City,State,Zip,Quick Draw,Latitude,Longitude,Georeference
0,117846,7-ELEVEN #47644H,2686 MIDDLE COUNTRY RD,LAKE GROVE,NY,11755,Y,40.858494,-73.107669,POINT (-73.107669 40.858494)
1,117828,CHITTENANGO STOP & SHOP LLC,298 GENESEE ST,CHITTENANGO,NY,13037,NaN,43.047743,-75.865717,POINT (-75.865717 43.047743)
2,117818,CVS PHARMACY #5048,1827 MAIN ST,PEEKSKILL,NY,10566,Y,41.295872,-73.894928,POINT (-73.894928 41.295872)
3,117817,BILTMORE FAMILY PHARMACY INC.,4638 MERRICK RD,MASSAPEQUA,NY,11758,NaN,40.666199,-73.463231,POINT (-73.463231 40.666199)
4,117816,BYRNE DAIRY #168 - VOLNEY,825 SOUTH 1ST STREET,FULTON,NY,13069,NaN,43.308864,-76.398637,POINT (-76.398637 43.308864)


In [6]:
retail.info()

<class 'pandas.DataFrame'>
RangeIndex: 13133 entries, 0 to 13132
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Retailer      13133 non-null  int64  
 1   Name          13133 non-null  str    
 2   Street        13133 non-null  str    
 3   City          13133 non-null  str    
 4   State         13133 non-null  str    
 5   Zip           13133 non-null  int64  
 6   Quick Draw    6164 non-null   str    
 7   Latitude      13133 non-null  float64
 8   Longitude     13133 non-null  float64
 9   Georeference  13133 non-null  str    
dtypes: float64(2), int64(2), str(6)
memory usage: 1.0 MB


### 3. Fix Data Points

Following retailers have incorrect locations:
- ELMIRAS STOP & SHOP
    - Updated coordinates
        - 42.089786
        - -76.818585
- IT'S FIVE O'CLOCK SOMEWHERE FI
    - Updated coordinates
        - 42.407394,
        - -74.615218


In [7]:
retail[retail["Name"] == "IT'S FIVE O'CLOCK SOMEWHERE FI"]

,Retailer,Name,Street,City,State,Zip,Quick Draw,Latitude,Longitude,Georeference
2505,114123,IT'S FIVE O'CLOCK SOMEWHERE FI,75 MAIN ST,STAMFORD,NY,12167,NaN,40.40762,-74.61523,POINT (-74.61523 40.40762)


In [8]:
retail.loc[retail["Name"] == "ELMIRAS STOP & SHOP", "Georeference"] = "POINT (-76.818585 42.089786)"
retail.loc[retail["Name"] == "FIVE O'CLOCK SOMEWHERE FI", "Georeference"] = "POINT (-74.615218 42.407394)"

retail[retail["Name"] == "IT'S FIVE O'CLOCK SOMEWHERE FI"]

,Retailer,Name,Street,City,State,Zip,Quick Draw,Latitude,Longitude,Georeference
2505,114123,IT'S FIVE O'CLOCK SOMEWHERE FI,75 MAIN ST,STAMFORD,NY,12167,NaN,40.40762,-74.61523,POINT (-74.61523 40.40762)


### 2a. Convert 'Georeference' into geometry Dtype

- .from_wkt
    - Converts from wkt format into a geometry type

In [9]:
retail['geometry'] = gpd.GeoSeries.from_wkt(retail["Georeference"])
retail.info()

<class 'pandas.DataFrame'>
RangeIndex: 13133 entries, 0 to 13132
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   Retailer      13133 non-null  int64   
 1   Name          13133 non-null  str     
 2   Street        13133 non-null  str     
 3   City          13133 non-null  str     
 4   State         13133 non-null  str     
 5   Zip           13133 non-null  int64   
 6   Quick Draw    6164 non-null   str     
 7   Latitude      13133 non-null  float64 
 8   Longitude     13133 non-null  float64 
 9   Georeference  13133 non-null  str     
 10  geometry      13133 non-null  geometry
dtypes: float64(2), geometry(1), int64(2), str(6)
memory usage: 1.1 MB


### 3. Convert 'retail' into GeoDataFrame (retail_gdf)

- gpd.GeoDataFrame
    - Converts a Pandas data frame into a GeoDataFrame

In [10]:
# Convert the DataFrame to a GeoDataFrame with GeoPandas

retail_gdf = gpd.GeoDataFrame(retail)
retail_gdf.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 13133 entries, 0 to 13132
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   Retailer      13133 non-null  int64   
 1   Name          13133 non-null  str     
 2   Street        13133 non-null  str     
 3   City          13133 non-null  str     
 4   State         13133 non-null  str     
 5   Zip           13133 non-null  int64   
 6   Quick Draw    6164 non-null   str     
 7   Latitude      13133 non-null  float64 
 8   Longitude     13133 non-null  float64 
 9   Georeference  13133 non-null  str     
 10  geometry      13133 non-null  geometry
dtypes: float64(2), geometry(1), int64(2), str(6)
memory usage: 1.1 MB


In [11]:
sd.head(1)

,SCHOOLDIST,geometry
0,SO MOUNTAIN,"POLYGON ((423174.308 4658625.221, 423127.151 4..."


## II. Intersections

### 1. Set sd and retail to the same CRS

In order to determine whether the geometry intersects, both objects must be set to the same Coordinate Reference System (CRS) value.

In [12]:
print(sd.crs)

EPSG:26918


- .set_crs
    - Tags the geometry values as being CRS values
- .to_crs
    - Converts the values to the specific CRS system

The crs for 'sd' is set to EPSG:26918
- North America: https://epsg.io/26918

In [13]:
# Set geometry values using .set_crs
# Convert them to sd's crs values with

retail_gdf = retail_gdf.set_crs("EPSG:4326")
retail_gdf = retail_gdf.to_crs(sd.crs)
# Will fail if rerun after converting

### 2. Join sd and retail based on geometry

Determine which school district it's joining by merging sd and retail geo dataframes
- .sjoin
    - Spatial joining to geodataframes
    - how = "left"
        - Which data frame is kept intact
    - predicate = "intersects"
        - How to compare the geometry

In [14]:
retail_sd = gpd.sjoin(
    retail_gdf, sd,
    how="left",
    predicate="intersects"
)

In [15]:
retail_sd.sample(3)

,Retailer,Name,Street,City,State,Zip,Quick Draw,Latitude,Longitude,Georeference,geometry,index_right,SCHOOLDIST
12468,32567,CUMBERLAND FARMS #1585,505 CAMPBELL AV,TROY,NY,12180,Y,42.707041,-73.678496,POINT (-73.678496 42.707041),POINT (608224.718 4729129.621),548.0,TROY
10291,66709,7-ELEVEN #23358,1001 NORTH COUNTRY ROAD,STONY BROOK,NY,11790,NaN,40.922900,-73.126800,POINT (-73.1268 40.9229),POINT (657726.465 4531887.52),147.0,THREE VILLAGE
9292,75750,TOPS MARKETS #700,100 MAIN ST,CORINTH,NY,12822,Y,43.246917,-73.834391,POINT (-73.834391 43.246917),POINT (594625.961 4788894.616),774.0,CORINTH


----

## III. Tally lottery retailers by school district

### 1. Convert retail_sd into a Pandas dataframe

- Drop the following columns
    - Geometry for school districts
    - Quick Draw
    - index_right
- Convert into a Pandas dataframe

In [16]:
retail_sd = retail_sd.drop(columns = ["Georeference", "Quick Draw", "index_right"])

retail_sd = pd.DataFrame(retail_sd)

### 2. Create a new column called "sd_label" based on "SCHOOLDIST" values

In [17]:
retail_sd["sd_label"] = retail_sd["SCHOOLDIST"]

### 3. Combine NYC school districts into new column

Overwrite sd_label using .str.contains
- Manhattan, Queens, Richmond (Staten Island), Bronx, and Brooklyn into a single "NYC" label

In [18]:
retail_sd.loc[retail_sd["SCHOOLDIST"].str.contains("Manhattan|Bronx|Queens|Brooklyn|Richmond", case=False, na=False), "sd_label"] = "NEW YORK CITY"

In [19]:
retail_sd["sd_label"].value_counts()

sd_label
NEW YORK CITY    4469
BUFFALO           198
ROCHESTER         186
YONKERS           127
SYRACUSE          111
                 ... 
HARTFORD            1
ARGYLE              1
LISBON              1
KEENE               1
KIRYAS JOEL         1
Name: count, Length: 657, dtype: int64

In [20]:
retail_sd[retail_sd["Name"] == "IT'S FIVE O'CLOCK SOMEWHERE FI"]

,Retailer,Name,Street,City,State,Zip,Latitude,Longitude,geometry,SCHOOLDIST,sd_label
2505,114123,IT'S FIVE O'CLOCK SOMEWHERE FI,75 MAIN ST,STAMFORD,NY,12167,40.40762,-74.61523,POINT (532647.765 4473071.709),NaN,NaN


### 4. Correcting the NaNs

Correct locations that did not intersect with geometry

In [21]:
retail_sd[retail_sd["sd_label"].isna()]

,Retailer,Name,Street,City,State,Zip,Latitude,Longitude,geometry,SCHOOLDIST,sd_label
2065,114882,CAPTAIN JACK'S GOOD TIME TAVER,8505 GREIG ST,SODUS POINT,NY,14555,43.270059,-76.979979,POINT (339321.784 4792708.467),NaN,NaN
2505,114123,IT'S FIVE O'CLOCK SOMEWHERE FI,75 MAIN ST,STAMFORD,NY,12167,40.407620,-74.615230,POINT (532647.765 4473071.709),NaN,NaN
3086,113049,BUFFALO EXPRESS,3095 BUFFALO RD,ROCHESTER,NY,14624,43.277043,-77.638047,POINT (285941.053 4794960.153),NaN,NaN


- Sodus Point = Sodus
- Stamford = Stamford
- Rochester = Gates-Chili

In [22]:
mask = retail_sd["Name"] == "CAPTAIN JACK'S GOOD TIME TAVER"
retail_sd.loc[mask, ["SCHOOLDIST", "sd_label"]] = "SODUS"

mask = retail_sd["Name"] == "IT'S FIVE O'CLOCK SOMEWHERE FI"
retail_sd.loc[mask, ["SCHOOLDIST", "sd_label"]] = "STAMFORD"

mask = retail_sd["Name"] == "BUFFALO EXPRESS"
retail_sd.loc[mask, ["SCHOOLDIST", "sd_label"]] = "GATES-CHILI"

In [23]:
# Checking if anymore needs correcting

retail_sd[retail_sd["sd_label"].isna()]

,Retailer,Name,Street,City,State,Zip,Latitude,Longitude,geometry,SCHOOLDIST,sd_label


### 5. Tally locations by school district

In [24]:
# Create a data frame to store the total retailer values

sd_tally = pd.DataFrame(retail_sd["sd_label"].value_counts())
sd_tally = sd_tally.set_axis(['total_retailers'], axis = 1)

sd_tally

,total_retailers
sd_label,
NEW YORK CITY,4469
BUFFALO,198
ROCHESTER,186
YONKERS,127
SYRACUSE,111
...,...
HARTFORD,1
ARGYLE,1
LISBON,1


----

In [25]:
mask = retail_sd["sd_label"] == "NEW YORK CITY"
retail_sd[mask]

,Retailer,Name,Street,City,State,Zip,Latitude,Longitude,geometry,SCHOOLDIST,sd_label
10,117791,1008 CLINTONVILLE MART LLC,1008 CLINTONVILLE ST,WHITESTONE,NY,11357,40.792681,-73.813041,POINT (600138.267 4516420.813),QUEENS 25,NEW YORK CITY
12,117788,THE CHECK CASHING PLACE #49,565 W 181ST STREET,NEW YORK,NY,10033,40.849178,-73.932606,POINT (589974.55 4522562.814),MANHATTAN 6,NEW YORK CITY
13,117785,7-ELEVEN #32127D,87-16 ASTORIA BLVD.,EAST ELMHURST,NY,11369,40.764071,-73.882013,POINT (594359.785 4513168.385),QUEENS 30,NEW YORK CITY
16,117778,786 CANDY STORE INC,141 E 96TH ST,NEW YORK,NY,10128,40.786123,-73.951066,POINT (588502.235 4515544.361),MANHATTAN 2,NEW YORK CITY
17,117774,SUPER WINE & LIQUOR INC,149 PARK AVE,BROOKLYN,NY,11205,40.696394,-73.973868,POINT (586694.756 4505561.038),BROOKLYN 13,NEW YORK CITY
...,...,...,...,...,...,...,...,...,...,...,...
13128,708,RUBEX DRUGS INC,81-19 41ST AVE,ELMHURST,NY,11373,40.745881,-73.884415,POINT (594182.718 4511146.571),QUEENS 24,NEW YORK CITY
13129,614,HILLSIDE PHARMACY,184-20 HILLSIDE AVE,JAMAICA,NY,11432,40.714045,-73.777803,POINT (603232.776 4507732.389),QUEENS 29,NEW YORK CITY
13130,529,LEE & LIN CO INC,59-18 MAIN ST,FLUSHING,NY,11355,40.743728,-73.825662,POINT (599146.21 4510972.278),QUEENS 25,NEW YORK CITY
13131,448,STATEN ISLAND UNIVERSITY HOSP,475 SEAVIEW AVENUE,STATEN ISLAND,NY,10305,40.584131,-74.086979,POINT (577267.571 4492993.858),RICHMOND 31,NEW YORK CITY
